## Setup
Initialize imports, reproducibility, and intervention utilities.


In [ ]:
from utils.motif import init_repro

seed = 42
init_repro(seed, deterministic=True)

import sys, os, time, pickle, copy, math
sys.path.append("./utils")

import torch
from transformers import (
    CLIPProcessor, CLIPModel, 
    CLIPVisionModelWithProjection, 
    CLIPTokenizer, CLIPTextModelWithProjection,
    AutoProcessor, AutoModel
)
import matplotlib.pyplot as plt
from utils.explanations import (
    render_explained_video_small_tl,
    explain_instance,
    plot_attention_heatmaps,
    print_explanation_with_labels,
)
import clip
import wandb

import numpy as np
from utils.video_embedder import VideoEmbedder, Create_Concepts
from utils.motif import MoTIF, CBMTransformer, mean_cbm

import core.vision_encoder.pe as pe
import core.vision_encoder.transforms as pe_transformer

from utils.interventions import remove_topk_important_concepts, keep_only_topk_important_concepts, intervention_topk_concepts, remove_random_concepts, insert_random_concepts, remove_topk_important_windows, remove_random_windows, leave_one_out_concepts, leave_one_in_concepts, keep_only_topk_important_local_concepts, remove_topk_important_local_concepts, intervention_topk_local_concepts, remove_random_local_concepts, insert_random_local_concepts, measure_attention_channel_effects, intervention_topk_attention_channels, intervention_random_attention_channels, plot_intervention_results, plot_compare_interventions


## Dataset And Model Config
Configure dataset, CLIP backbone, concepts, and evaluation settings.


In [ ]:
num_epochs = 200
batch_size = 32
lse_tau = 1.0  # 0.5, 1.0, 2.0, 4.0
l1_lambda = 1e-4  # 1e-2, 1e-3, 1e-4, 1e-5
lambda_sparse = 1e-4  # 1e-2, 1e-3, 1e-4, 1e-5
lr = 1e-4  # 1e-2, 1e-3, 1e-4, 1e-5
transformer_layers = 1  # 1,2,4
diagonal_attention = True  # True or False
enforce_nonneg = True  # True or False
class_weights = True  # True or False
weight_decay = 1e-2
d = 1
stride = 0

test_split = "s1"  # s1, s2, s3, s4xw
window_size = 32  # 8,16,32,64

dataset = "breakfast"
random = True
use_wandb = False

clip_model = "res50"  # res50, b16, b32, l14, siglip, clip4clip

if dataset == "breakfast":
    dataset_name = "Breakfast"
elif dataset == "ucf101":
    dataset_name = "UCF101"
elif dataset == "hmdb51":
    dataset_name = "HMDB"
elif dataset == "something2":
    dataset_name = "Something2"

num_epochs= 50
batch_size = 512
lse_tau = 1.0 # 0.5, 1.0, 2.0, 4.0
l1_lambda=1e-4 # 1e-2, 1e-3, 1e-4, 1e-5
lambda_sparse=1e-4 # 1e-2, 1e-3, 1e-4, 1e-5
lr = 1e-3 # 1e-2, 1e-3, 1e-4, 1e-5
transformer_layers= 1 #1,2,4
diagonal_attention = True # True or False
enforce_nonneg=True # True or False
class_weights=True # True or False
weight_decay = 1e-2
d = 1

test_split = "s1" # s1, s2, s3, s4xw
window_size = 32 #8,16,32,64
train_window_len = 64  
stride = 0

dataset = "breakfast"
random = True

clip_model = "res50" # res50, b16, b32, l14, siglip, clip4clip

if dataset == "breakfast":
    dataset_name = "Breakfast"
elif dataset == "ucf101":
    dataset_name = "UCF101"
elif dataset == "hmdb51":
    dataset_name = "HMDB" 
elif dataset == "something2":
    dataset_name = "Something2" 
    

folder_path = [f"./Datasets/{dataset_name}/Video_data"]
output_dir = "./Embeddings/Datasets" 


# Align and load CLIP or related models
if clip_model == "b32":
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval()
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", use_fast=False)
    embedd_path = f"./Embeddings/Videos/{dataset_name}/{random}_{window_size}_clip_b32.pkl"
    clip_name = "clip"
elif clip_model == "b16":
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").eval()
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16", use_fast=False)
    embedd_path = f"./Embeddings/Videos/{dataset_name}/{random}_{window_size}_clip_b16.pkl"
    clip_name = "clip"
elif clip_model == "l14":
    model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").eval()
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14", use_fast=False)
    embedd_path = f"./Embeddings/Videos/{dataset_name}/{random}_{window_size}_clip_l14.pkl"
    clip_name = "clip"
elif clip_model == "res50":
    model, preprocess = clip.load("RN50", device="cpu")
    processor = preprocess
    embedd_path = f"./Embeddings/Videos/{dataset_name}/{random}_{window_size}_clip_res50.pkl"
    clip_name = "res50"
elif clip_model == "clip4clip":
    model = CLIPVisionModelWithProjection.from_pretrained("Searchium-ai/clip4clip-webvid150k").eval()
    model_text = CLIPTextModelWithProjection.from_pretrained("Searchium-ai/clip4clip-webvid150k")
    processor = CLIPTokenizer.from_pretrained("Searchium-ai/clip4clip-webvid150k")
    embedd_path = f"./Embeddings/Videos/{dataset_name}/{random}_{window_size}_clip_clip4clip.pkl"
    clip_name = "clip4clip"
elif clip_model == "siglip":
    model = AutoModel.from_pretrained("google/siglip-base-patch16-224")
    processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
    clip_name = "siglip"
    embedd_path = f"./Embeddings/Videos/{dataset_name}/{random}_{window_size}_clip_siglip.pkl"
elif clip_model == "siglipl14":
    model = AutoModel.from_pretrained("google/siglip-so400m-patch14-384")
    processor = AutoProcessor.from_pretrained("google/siglip-so400m-patch14-384")
    clip_name = "siglipl14"
    embedd_path = f"./Embeddings/Videos/{dataset_name}/{random}_{window_size}_clip_siglipl14.pkl"
elif clip_model == "pe-l14":
    model = pe.CLIP.from_config("PE-Core-L14-336", pretrained=True)
    processor = pe_transformer.get_image_transform(model.image_size)
    tokenizer = pe_transformer.get_text_tokenizer(model.context_length)
    clip_name = "pe-l14"
    embedd_path = f"./Embeddings/Videos/{dataset_name}/{random}_{window_size}_clip_pe-l14.pkl"
    if stride > 0:
        embedd_path = f"./Embeddings/Videos/{dataset_name}/{random}_{window_size}_{stride}_clip_pe-l14.pkl"
else:
    model = None
    processor = None
    model_text = None
    raise ValueError(f"Unknown clip_model {clip_model}")

# Align embedder
embedder = VideoEmbedder(clip_name, model, processor)
embedder.dataset_name = dataset

if os.path.exists(embedd_path):
    try:
        with open(embedd_path, "rb") as f:
            embedder = pickle.load(f)
            print("Loaded existing embedder from", embedd_path)
    except FileNotFoundError:
        print("Embedder file not found, creating a new one.")
else:
    embedder.process_data(
        folder_path,
        window_size=window_size,
        output_path=output_dir,
        save_intermediate=False,
    )

    with open(embedd_path, "wb") as f:
        pickle.dump(embedder, f)

if clip_model == "clip4clip":
    concepts = Create_Concepts(clip_name, model_text, processor)
elif clip_model == "pe-l14":
    concepts = Create_Concepts(clip_name, model, tokenizer)
else:
    concepts = Create_Concepts(clip_name, model, processor)

if dataset == "breakfast":
    text_concepts = [
        "grind, fill, boil, pour, steep, brew, tamp, insert, steam, froth, stir, sip, add, slice, toast, butter, spread, cut, assemble, grate, chop, peel, core, squeeze, pit, mash, crack, whisk, beat, fry, scramble, flip, mix, cook, drizzle, serve, drain, grill, preheat, bake, warm, wash, rinse, blend, measure, set, open, close, take, put, remove, pack, dry, wipe, sit, stand, carry, pick, blow, taste, adjust, reach, place, seal, unwrap, unscrew, scoop, zest, juice, start, stop, turn, heat, cool, toss, shake, tap, knock, press, release, slide, rotate, fold, unfold, wring, sprinkle, arrange, sort, stack, unstack, hide, reveal, cover, uncover, balance, tilt, catch, throw, drop, roll, toss, spin, twist, poke, pinch, pull, push, drag, scrub, brush, comb, shave, zip, button, tie, untie, snap, clap, wave, point, nod, gesture, smile, frown, laugh, coffee, kettle, water, tea, milk, sugar, cereal, yogurt, granola, fruit, bread, bagel, cheese, tomato, cucumber, onion, herb, banana, apple, orange, avocado, egg, bacon, sausage, ham, pan, stove, oven, pastry, croissant, strawberry, blender, ice, batter, syrup, cinnamon, honey, jar, plate, cup, spoon, fork, knife, tongs, lid, package, container, carton, bottle, pantry, fridge, cupboard, counter, sink, dish, towel, timer, mug, bowl, spatula, ladle, grater, peeler, colander, sieve, cuttingboard, tray, ovenmitt, scale, thermometer, stool, chair, table, napkin, freezer, hood, burner, flame, plug, socket, switch, knob, handle, cover, stirrer, measuringcup, measuringspoon, recipe, cookbook, ingredient, serving, leftover, waste, soap, sponge, detergent, faucet, garbage, recycle, bin"
    ]  # shorter for test
elif dataset == "ucf101":
    text_concepts = [
        "jump, swing, skip, throw, catch, dribble, bounce, kick, pass, hit, serve, smash, block, spike, dive, swim, climb, grab, pull, hang, push, sit, ride, pedal, balance, stop, start, steer, mount, dismount, gallop, control, lift, curl, press, squat, deadlift, jab, hook, uppercut, dodge, wrestle, grapple, flip, perform, walk, handstand, run, sprint, shoot, turn, grind, row, paddle, surf, stand, tuck, enter, splash, wave, clap, raise, squat, spin, dance, breakdance, strike, parry, fight, reload, aim, release, bowl, swing, pitch, hit, catch, skateboard, snowboard, ski, trampoline, yoga, sword, gun, archery, hockey, basketball, volleyball, soccer, rugby, baseball, cricket, rope, ball, bat, racket, puck, stick, net, goal, pool, lane, wall, ladder, bar, dumbbell, barbell, mat, beam, hurdle, bicycle, helmet, horse, reins, rail, snowboard, skis, kayak, canoe, paddle, surfboard, gloves, boxing, stage, microphone, instrument, music, sheet, player, opponent, teammate, referee, coach, dancer, athlete, gymnast, swimmer, skater, snowboarder, skateboarder, rower, surfer, archer, shooter, bow, club, frisbee, arrow, target, goalpost, jersey, uniform, cap, helmet, pad, netting, court, field, track, floor, platform, water, sand, snow, ice, gym, stadium, arena, ring, mat, beam, hoop, basket, scoreboard, timer"
    ]
elif dataset == "hmdb51":
    text_concepts = [
        "bow, fight, sword, walk, run, sprint, jog, stand, up, sit, down, jump, hop, leap, fall, roll, crouch, bend, stretch, turn, around, look, up, look, down, look, left, look, right, nod, head, shake, head, smile, laugh, frown, yawn, talk, mouth, words, sing, chew, eat, with, hands, eat, with, utensils, drink, from, cup, drink, from, bottle, sip, blow, kiss, hug, wave, hand, point, reach, grab, object, release, object, throw, object, catch, object, toss, ball, kick, ball, hit, with, hand, punch, block, push, pull, lift, object, carry, object, drag, object, drop, object, catch, fall, climb, up, climb, down, crawl, swim, dive, surface, float, balance, ride, bicycle, pedal, bicycle, brake, bicycle, steer, bicycle, mount, horse, dismount, horse, gallop, ride, skateboard, skate, sled, ski, snowboard, slide, skate, backward, turn, skateboard, shoot, basketball, dribble, ball, bounce, ball, serve, tennis, swing, racket, hit, tennis, ball, swing, bat, hit, baseball, throw, frisbee, catch, frisbee, juggle, spin, object, roll, ball, kick, leg, high, kick, leg, low, flip, somersault, cartwheel, handstand, headstand, touch, head, touch, face, wash, face, comb, hair, brush, hair, brush, teeth, shave, apply, makeup, put, on, hat, take, off, hat, put, on, jacket, take, off, jacket, button, shirt, zip, jacket, tie, shoelace, untie, shoelace, open, door, close, door, knock, door, enter, room, exit, room, sit, on, chair, stand, from, chair, lie, down, wake, up, sleep, sprint, start, cross, finish, line"
    ]
elif dataset == "something2":
    text_concepts = [
        "push, pull, lift, drop, hold, carry, throw, catch, slide, drag, roll, spin, rotate, flip, fold, unfold, wrap, unwrap, tie, untie, fasten, unfasten, tighten, loosen, break, cut, slice, chop, tear, peel, crumple, flatten, bend, stretch, shake, stir, pour, scoop, sprinkle, stack, unstack, assemble, disassemble, open, close, lock, unlock, press, tap, swipe, scroll, zoom in, zoom out, point, touch, wave, clap, knock, snap, swing, juggle, bounce, balance, topple, insert, remove, fill, empty, mix, separate, spill, scatter, gather, cover, uncover, hide, reveal, lean, tilt, climb, crawl, jump, hop, walk, run, sprint, stumble, fall, get up, sit, stand, kneel, crouch, bow, dance, spin dance, nod, shake head, smile, frown, laugh, cry, shout, whisper, speak, yawn, sneeze, cough, sleep, wake, eat, chew, bite, sip, drink, spit, blow, smell, taste, write, draw, erase, paint, type, click, drag mouse, plug, unplug, connect, disconnect, turn on, turn off, start, stop, accelerate, decelerate, pretend to push, pretend to pull, pretend to pour, pretend to eat, pretend to drink, pretend to throw, pretend to catch, pretend to type, pretend to swipe, pretend to scroll, pretend to climb, pretend to fall, pretend to hug, pretend to kiss, pretend to wave, pretend to play guitar, pretend to drive, pretend to steer, pretend to read, pretend to sleep, pretend to wake, pretend to write, pretend to draw, pretend to paint, pretend to clean, pretend to cook, pretend to stir, pretend to measure, pretend to weigh, pretend to look around, pretend to search, pretend to point, pretend to balance, pretend to open, pretend to close, pretend to lock, pretend to unlock, pretend to kick, pretend to punch, pretend to block, pretend to dodge, pretend to jump rope, pretend to row, pretend to paddle, pretend to shoot arrow, pretend to load gun, pretend to fire gun, pretend to throw ball, pretend to dribble, pretend to shoot basket, pretend to swing bat, pretend to serve, pretend to catch fish, pretend to steer wheel, pretend to honk, pretend to use controller, pretend to play piano, pretend to play drums, pretend to dance, pretend to sing, pretend to clap, pretend to salute, pretend to bow, pretend to shake hands, pretend to hug, pretend to kiss, object, container, box, cup, bowl, plate, spoon, knife, fork, chopstick, pen, pencil, paper, book, phone, remote, laptop, keyboard, mouse, bag, backpack, toy, ball, fruit, apple, orange, banana, grape, vegetable, carrot, cucumber, tomato, bottle, can, lid, cap, key, lock, door, window, wall, floor, table, chair, shelf, hand, finger, arm, face, person, other, background, surface, inside, outside, top, bottom, left, right, upward, downward, hot, cold, wet, dry, clean, dirty, empty, full, broken, fixed, smooth, rough, heavy, light, fragile, durable, rollable, stackable, squeezable, pourable, spillable, openable, closeable, edible, drinkable"
    ]

else:
    raise ValueError("Unknown dataset")

concepts.embedd_text(text_concepts)

## Load Trained CBM
Load the trained MoTIF / CBM checkpoint used for interventions.


In [ ]:
model_name = f"./Models/{dataset_name}_{clip_model}_testing_{"1.0"}_{window_size}_motif.pkl"

print(model_name)
if os.path.exists(model_name):
    try:
        with open(model_name, "rb") as f:
            cbm_model = pickle.load(f)
            print("Loaded existing cbm_model from", model_name)
    except FileNotFoundError:
        print("cbm_model file not found, creating a new one.")
else:
    print("Model not found", model_name)

## Global Concept Interventions
Compute keep-only and remove-top-k interventions over global concept channels.


In [ ]:
k_values = (0, 1, 2, 3, 4, 5, 10)

results_keep = keep_only_topk_important_concepts(
    cbm_model=cbm_model,
    explain_instance_fn=explain_instance,
    k_values=k_values,
    concept_order_mode="positive",   # or "absolute"
    device="cuda:0",
)

results_remove = remove_topk_important_concepts(
    cbm_model=cbm_model,
    explain_instance_fn=explain_instance,
    k_values=k_values,
    concept_order_mode="positive",   # or "absolute"
    device="cuda:0",
)

print("KEEP ONLY:", results_keep)
print("REMOVE TOP-K:", results_remove)

## Global Concept Plots
Plot the global top-k intervention curves.


In [ ]:
plot_intervention_results(
    results_keep,
    metrics=["pred_overlap", "accuracy", "confidence_drop_original_pred_class"],
    title="Keep only top-k important concepts",
)

plot_intervention_results(
    results_remove,
    metrics=["pred_overlap", "accuracy", "confidence_drop_original_pred_class"],
    title="Remove top-k important concepts",
)

## Global Comparison
Compare global keep-only and remove-top-k interventions directly.


In [ ]:
plot_compare_interventions(
    results_keep,
    results_remove,
    metric="confidence_drop_original_pred_class",
)

## Local Concept Interventions
Intervene on local (time window, concept) slots using top-k importance.


In [ ]:
# Local concept slots = specific (time window, concept) pairs from concept_contributions_per_time
k_values_local = (0, 1, 2, 3, 4, 5, 10, 20, 50)

results_keep_local = keep_only_topk_important_local_concepts(
    cbm_model=cbm_model,
    explain_instance_fn=explain_instance,
    k_values=k_values_local,
    concept_order_mode="positive",   # or "absolute"
    device="cuda:0",
)

results_remove_local = remove_topk_important_local_concepts(
    cbm_model=cbm_model,
    explain_instance_fn=explain_instance,
    k_values=k_values_local,
    concept_order_mode="positive",   # or "absolute"
    device="cuda:0",
)

print("KEEP ONLY LOCAL:", results_keep_local)
print("REMOVE TOP-K LOCAL:", results_remove_local)

plot_intervention_results(
    results_keep_local,
    metrics=["pred_overlap", "accuracy", "confidence_drop_original_pred_class"],
    title="Keep only top-k important local concepts",
)

plot_intervention_results(
    results_remove_local,
    metrics=["pred_overlap", "accuracy", "confidence_drop_original_pred_class"],
    title="Remove top-k important local concepts",
)

plot_compare_interventions(
    results_keep_local,
    results_remove_local,
    metric="confidence_drop_original_pred_class",
)


## Random Baselines
Run random removal and random insertion baselines for global and local concepts.


In [ ]:
num_trials_random = 10
random_seed = 42

results_remove_random = remove_random_concepts(
    cbm_model=cbm_model,
    explain_instance_fn=explain_instance,
    k_values=k_values,
    num_trials=num_trials_random,
    random_seed=random_seed,
    device="cuda:0",
)

results_insert_random = insert_random_concepts(
    cbm_model=cbm_model,
    explain_instance_fn=explain_instance,
    k_values=k_values,
    num_trials=num_trials_random,
    random_seed=random_seed,
    device="cuda:0",
)

print("REMOVE RANDOM:", results_remove_random)
print("INSERT RANDOM:", results_insert_random)

plot_intervention_results(
    results_remove_random,
    metrics=["pred_overlap", "accuracy", "confidence_drop_original_pred_class"],
    title="Remove k random concepts",
)

plot_intervention_results(
    results_insert_random,
    metrics=["pred_overlap", "accuracy", "confidence_drop_original_pred_class"],
    title="Insert noise into k random concepts",
)

plot_compare_interventions(
    results_remove,
    results_remove_random,
    metric="accuracy",
    label_a="remove_topk",
    label_b="remove_random",
    title="Global removal: top-k vs random",
)

plot_compare_interventions(
    results_remove_random,
    results_insert_random,
    metric="accuracy",
    label_a="remove_random",
    label_b="insert_random",
    title="Global random removal vs random insertion",
)

results_remove_random_local = remove_random_local_concepts(
    cbm_model=cbm_model,
    explain_instance_fn=explain_instance,
    k_values=k_values_local,
    num_trials=num_trials_random,
    random_seed=random_seed,
    device="cuda:0",
)

results_insert_random_local = insert_random_local_concepts(
    cbm_model=cbm_model,
    explain_instance_fn=explain_instance,
    k_values=k_values_local,
    num_trials=num_trials_random,
    random_seed=random_seed,
    device="cuda:0",
)

print("REMOVE RANDOM LOCAL:", results_remove_random_local)
print("INSERT RANDOM LOCAL:", results_insert_random_local)

plot_intervention_results(
    results_remove_random_local,
    metrics=["pred_overlap", "accuracy", "confidence_drop_original_pred_class"],
    title="Remove k random local concepts",
)

plot_intervention_results(
    results_insert_random_local,
    metrics=["pred_overlap", "accuracy", "confidence_drop_original_pred_class"],
    title="Insert noise into k random local concepts",
)

plot_compare_interventions(
    results_remove_local,
    results_remove_random_local,
    metric="accuracy",
    label_a="remove_topk_local",
    label_b="remove_random_local",
    title="Local removal: top-k vs random",
)

plot_compare_interventions(
    results_remove_random_local,
    results_insert_random_local,
    metric="accuracy",
    label_a="remove_random_local",
    label_b="insert_random_local",
    title="Local random removal vs random insertion",
)


## Attention Channel Interventions
Measure per-channel attention effects and compare top-k versus random attention neutralization.


In [ ]:
attention_score_mode = "diagonal_mass"   # or "low_entropy", "top_time_mass"
attention_intervention_mode = "uniform"  # or "identity"

attention_channel_effects = measure_attention_channel_effects(
    cbm_model=cbm_model,
    explain_instance_fn=explain_instance,
    attn_mode=attention_intervention_mode,
    layer_agg="mean",
    attention_score_mode=attention_score_mode,
    device="cuda:0",
)

top_attention_channels = sorted(
    attention_channel_effects.items(),
    key=lambda kv: kv[1]["confidence_drop_original_pred_class"],
    reverse=True,
)[:10]

print("Top attention channels by single-channel causal effect:")
for channel_idx, stats in top_attention_channels:
    print(
        channel_idx,
        {
            "confidence_drop_original_pred_class": round(stats["confidence_drop_original_pred_class"], 6),
            "pred_overlap": round(stats["pred_overlap"], 6),
            "attention_score": round(stats["attention_score"], 6),
        },
    )

k_values_attention = (0, 1, 2, 3, 4, 5, 10)

results_remove_attention_topk = intervention_topk_attention_channels(
    cbm_model=cbm_model,
    explain_instance_fn=explain_instance,
    k_values=k_values_attention,
    attention_score_mode=attention_score_mode,
    attn_mode=attention_intervention_mode,
    layer_agg="mean",
    device="cuda:0",
)

results_remove_attention_random = intervention_random_attention_channels(
    cbm_model=cbm_model,
    explain_instance_fn=explain_instance,
    k_values=k_values_attention,
    attn_mode=attention_intervention_mode,
    num_trials=num_trials_random,
    random_seed=random_seed,
    device="cuda:0",
)

print("REMOVE TOP-K ATTENTION:", results_remove_attention_topk)
print("REMOVE RANDOM ATTENTION:", results_remove_attention_random)

plot_intervention_results(
    results_remove_attention_topk,
    metrics=["pred_overlap", "accuracy", "confidence_drop_original_pred_class"],
    title=f"Neutralize top-k attention channels ({attention_intervention_mode})",
)

plot_intervention_results(
    results_remove_attention_random,
    metrics=["pred_overlap", "accuracy", "confidence_drop_original_pred_class"],
    title=f"Neutralize k random attention channels ({attention_intervention_mode})",
)

plot_compare_interventions(
    results_remove_attention_topk,
    results_remove_attention_random,
    metric="accuracy",
    label_a="topk_attention",
    label_b="random_attention",
    title=f"Attention interventions: top-k vs random ({attention_intervention_mode})",
)


## Window And Leave-One Analysis
Evaluate window removal, leave-one-out / leave-one-in concepts, and attention shuffle / reverse baselines.


In [ ]:
results_remove_windows_topk = remove_topk_important_windows(
    cbm_model=cbm_model,
    explain_instance_fn=explain_instance,
    k_values=k_values_attention,
    device="cuda:0",
)

results_remove_windows_random = remove_random_windows(
    cbm_model=cbm_model,
    explain_instance_fn=explain_instance,
    k_values=k_values_attention,
    num_trials=num_trials_random,
    random_seed=random_seed,
    device="cuda:0",
)

print("REMOVE TOP-K WINDOWS:", results_remove_windows_topk)
print("REMOVE RANDOM WINDOWS:", results_remove_windows_random)

plot_intervention_results(
    results_remove_windows_topk,
    metrics=["pred_overlap", "accuracy", "confidence_drop_original_pred_class"],
    title="Remove top-k important windows",
)

plot_intervention_results(
    results_remove_windows_random,
    metrics=["pred_overlap", "accuracy", "confidence_drop_original_pred_class"],
    title="Remove k random windows",
)

plot_compare_interventions(
    results_remove_windows_topk,
    results_remove_windows_random,
    metric="accuracy",
    label_a="remove_topk_windows",
    label_b="remove_random_windows",
    title="Window removal: top-k vs random",
)

leave_one_out_results = leave_one_out_concepts(
    cbm_model=cbm_model,
    explain_instance_fn=explain_instance,
    device="cuda:0",
)

leave_one_in_results = leave_one_in_concepts(
    cbm_model=cbm_model,
    explain_instance_fn=explain_instance,
    device="cuda:0",
)

concept_names = text_concepts if len(text_concepts) == len(leave_one_out_results) else [f"c{i}" for i in range(len(leave_one_out_results))]

top_loo = sorted(
    leave_one_out_results.items(),
    key=lambda kv: kv[1]["confidence_drop_original_pred_class"],
    reverse=True,
)[:10]

top_loi = sorted(
    leave_one_in_results.items(),
    key=lambda kv: kv[1]["accuracy"],
    reverse=True,
)[:10]

print("Top leave-one-out concepts:")
for concept_idx, stats in top_loo:
    print(
        concept_idx,
        concept_names[concept_idx],
        {
            "confidence_drop_original_pred_class": round(stats["confidence_drop_original_pred_class"], 6),
            "pred_overlap": round(stats["pred_overlap"], 6),
            "concept_score": round(stats["concept_score"], 6),
        },
    )

print("Top leave-one-in concepts:")
for concept_idx, stats in top_loi:
    print(
        concept_idx,
        concept_names[concept_idx],
        {
            "accuracy": round(stats["accuracy"], 6),
            "pred_overlap": round(stats["pred_overlap"], 6),
            "concept_score": round(stats["concept_score"], 6),
        },
    )

results_remove_attention_topk_reverse = intervention_topk_attention_channels(
    cbm_model=cbm_model,
    explain_instance_fn=explain_instance,
    k_values=k_values_attention,
    attention_score_mode=attention_score_mode,
    attn_mode="reverse",
    layer_agg="mean",
    device="cuda:0",
)

results_remove_attention_topk_shuffle = intervention_topk_attention_channels(
    cbm_model=cbm_model,
    explain_instance_fn=explain_instance,
    k_values=k_values_attention,
    attention_score_mode=attention_score_mode,
    attn_mode="shuffle",
    layer_agg="mean",
    device="cuda:0",
)

print("REMOVE TOP-K ATTENTION REVERSE:", results_remove_attention_topk_reverse)
print("REMOVE TOP-K ATTENTION SHUFFLE:", results_remove_attention_topk_shuffle)

plot_compare_interventions(
    results_remove_attention_topk,
    results_remove_attention_topk_reverse,
    metric="accuracy",
    label_a="uniform_attention",
    label_b="reverse_attention",
    title="Top-k attention: uniform vs reverse",
)

plot_compare_interventions(
    results_remove_attention_topk,
    results_remove_attention_topk_shuffle,
    metric="accuracy",
    label_a="uniform_attention",
    label_b="shuffle_attention",
    title="Top-k attention: uniform vs shuffle",
)
